# 04 — Feature engineering
### Atlantic Haven Hotels — Prédiction d'annulation de réservation

Ce notebook correspond à l'**étape 4** du sujet : création de variables pertinentes à partir des
dates, du séjour, du prix et de l'historique client, sans fuite de cible, avec un **gain démontré
expérimentalement**.

**Démarche suivie** : on part d'un jeu large de variables candidates, on mesure honnêtement leur effet
(y compris quand il est négatif ou nul), puis on réduit à un sous-ensemble dont le gain est réel et
reproductible — plutôt que de garder toutes les variables créées sans vérification.

## 0. Configuration et chargement des données

In [1]:
import sys, os, json
for _candidate in (".", ".."):
    if os.path.isdir(os.path.join(_candidate, "src")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from src.config import (RANDOM_STATE, NUM_COLS, CAT_COLS, NUM_COLS_FE, CAT_COLS_FE,
                         FE_NUM_COLS, FE_NUM_COLS_CANDIDATES, FE_CAT_COLS_CANDIDATES, ARTIFACTS_DIR)
from src.data import load_train_test, temporal_split
from src.preprocessing import prepare_features, build_preprocessor, make_pipeline
from src.features import add_engineered_features, GroupRelativePriceTransformer
from src.evaluation import evaluate, threshold_curve, plot_threshold_curve
from src.model_io import save_model

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

train, test, data_dict = load_train_test()
X_tr, X_val, y_tr, y_val, split_date = temporal_split(train)
X_tr_p, X_val_p = prepare_features(X_tr), prepare_features(X_val)
X_tr_fe, X_val_fe = prepare_features(add_engineered_features(X_tr)), prepare_features(add_engineered_features(X_val))
print(f"Train interne : {X_tr.shape[0]} lignes | Validation : {X_val.shape[0]} lignes")

Train interne : 6400 lignes | Validation : 1600 lignes


## 4.1 Variables candidates créées

| Variable | Construction | Justification |
|---|---|---|
| `mois_arrivee` | mois de `date_arrivee` | Saisonnalité observée en EDA §1.9 |
| `jour_semaine_arrivee` | jour de semaine de `date_arrivee` | Granularité plus fine que `arrivee_weekend` |
| `delai_log` | `log1p(delai_reservation_jours)` | Compresse la relation non linéaire observée en EDA §1.6 |
| `delai_bin` | tranches `[0-7j, 8-30j, 31-90j, 91-180j, 181j+]` | Capture les paliers de risque identifiés en EDA |
| `personnes_totales` | `adultes + enfants` (NA→0) | Taille réelle du groupe |
| `personnes_par_chambre` | `personnes_totales / chambres` | Densité d'occupation, proxy de type de séjour |
| `nuits_par_chambre` | `nuits / chambres` | Proxy de durée de séjour par unité louée |
| `a_enfants` | `enfants > 0` | Signal binaire simple |
| `prix_total_par_personne` | `montant_total_eur / personnes_totales` | Normalise le montant par la taille du groupe |
| `prix_relatif_destination` | `prix_moyen_nuit_eur / prix moyen du type de destination` (appris sur train) | Prix élevé *relativement à sa destination* |
| `deja_annule_avant` | `annulations_passees > 0` | Signal binaire d'historique |
| `taux_annulation_passee` | `annulations_passees / reservations_passees` (0 si aucun historique) | Comportement relatif du client |
| `client_nouveau_sans_historique` | `reservations_passees == 0` | Isole les clients sans historique |

**Sur la fuite de cible :** toutes ces variables sont des fonctions déterministes des colonnes brutes
disponibles *au moment de la réservation*. Aucune n'utilise `reservation_annulee`. La seule variable
nécessitant une statistique apprise (`prix_relatif_destination`) est encapsulée dans un transformer
scikit-learn (`GroupRelativePriceTransformer`) dont le `.fit()` n'est appelé que sur le train.

In [2]:
engineered_preview = add_engineered_features(X_tr)
engineered_preview[["mois_arrivee", "jour_semaine_arrivee", "delai_log", "delai_bin",
                     "personnes_totales", "personnes_par_chambre", "a_enfants",
                     "prix_total_par_personne", "deja_annule_avant", "taux_annulation_passee",
                     "client_nouveau_sans_historique"]].head()

,mois_arrivee,jour_semaine_arrivee,delai_log,delai_bin,personnes_totales,personnes_par_chambre,a_enfants,prix_total_par_personne,deja_annule_avant,taux_annulation_passee,client_nouveau_sans_historique
0,3,0,4.276666,31-90j,2.0,2.0,0,321.495000,0,0.000000,1
1,1,5,1.945910,0-7j,3.0,3.0,1,187.000000,1,0.111111,0
2,3,6,4.442651,31-90j,2.0,2.0,0,268.205000,0,0.000000,1
3,1,0,2.197225,8-30j,3.0,3.0,1,187.953333,0,0.000000,1
4,1,2,2.397895,8-30j,2.0,2.0,0,266.670000,0,0.000000,1


## 4.2 Premier test : jeu complet de variables — un gain ?

On compare, à hyperparamètres identiques (mêmes réglages que la baseline et que le XGBoost retenu à
l'étape 3), le F1 obtenu **avec toutes les variables candidates** vs sans. Le seuil est ré-optimisé pour
chaque configuration.

In [3]:
def eval_pipeline(pipe, X_tr_in, X_val_in, label):
    pipe.fit(X_tr_in, y_tr)
    proba = pipe.predict_proba(X_val_in)[:, 1]
    thresholds, f1s, _, _ = threshold_curve(y_val, proba)
    best_t = thresholds[np.argmax(f1s)]
    res, _ = evaluate(y_val, proba, threshold=best_t, label=label)
    return res, proba

with open(ARTIFACTS_DIR / "xgboost_v1.json") as f:
    xgb_meta = json.load(f)
xgb_best_params = {k.replace("model__", ""): v for k, v in xgb_meta["best_params"].items()}
scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

def make_xgb():
    return XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
                          eval_metric="logloss", n_jobs=-1, **xgb_best_params)

def make_logreg():
    return LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)

full_num = NUM_COLS + FE_NUM_COLS_CANDIDATES
full_cat = CAT_COLS + FE_CAT_COLS_CANDIDATES

def make_fe_pipeline(estimator, num_cols, cat_cols):
    return Pipeline([
        ("price_relative", GroupRelativePriceTransformer()),
        ("preprocess", build_preprocessor(num_cols, cat_cols)),
        ("model", estimator),
    ])

res_lr_base, _ = eval_pipeline(make_pipeline(make_logreg()), X_tr_p, X_val_p, "LR — sans FE")
res_lr_full, _ = eval_pipeline(make_fe_pipeline(make_logreg(), full_num, full_cat), X_tr_fe, X_val_fe, "LR — FE complet")
res_xgb_base, _ = eval_pipeline(make_pipeline(make_xgb()), X_tr_p, X_val_p, "XGBoost — sans FE")
res_xgb_full, _ = eval_pipeline(make_fe_pipeline(make_xgb(), full_num, full_cat), X_tr_fe, X_val_fe, "XGBoost — FE complet")

premier_test = pd.DataFrame([res_lr_base, res_lr_full, res_xgb_base, res_xgb_full]).set_index("modele")
premier_test[["seuil", "f1", "precision", "recall", "roc_auc"]].round(4)

,seuil,f1,precision,recall,roc_auc
modele,,,,,
LR — sans FE,0.300,0.4727,0.3183,0.9184,0.6532
LR — FE complet,0.350,0.4707,0.3227,0.8695,0.6523
XGBoost — sans FE,0.425,0.4842,0.3468,0.8019,0.6565
XGBoost — FE complet,0.425,0.4824,0.3428,0.8135,0.6549


**Constat inattendu :** le jeu complet de variables candidates **n'améliore ni la régression logistique
ni XGBoost** — le F1 et le ROC-AUC baissent légèrement pour les deux modèles. Plutôt que d'ignorer ce
résultat ou de chercher un seuil qui « sauve » le chiffre, on l'investigue : plusieurs variables sont
**redondantes** entre elles (`delai_reservation_jours`, `delai_log` et `delai_bin` encodent la même
information sous trois formes) ou redondantes avec les colonnes brutes déjà présentes (`a_enfants` avec
`enfants`, `deja_annule_avant` et `client_nouveau_sans_historique` avec
`annulations_passees`/`reservations_passees`). Sur 6400 lignes d'entraînement, ce surplus de dimensions
corrélées ajoute du bruit plutôt que de l'information nouvelle.

## 4.3 Ablation : quel sous-ensemble apporte un vrai gain ?

On teste des sous-ensembles ciblés, par thème (temporel / séjour-prix / historique), pour identifier les
variables qui apportent une information **réellement nouvelle** (non redondante avec les colonnes brutes
ou entre elles).

In [4]:
ablation_sets = {
    "Temporel seul (mois, jour_semaine, delai_log, delai_bin)":
        (["mois_arrivee", "jour_semaine_arrivee", "delai_log"], ["delai_bin"]),
    "Séjour + prix (personnes_par_chambre, nuits_par_chambre, prix_total_par_personne, prix_relatif_destination)":
        (["personnes_par_chambre", "nuits_par_chambre", "prix_total_par_personne", "prix_relatif_destination"], []),
    "Historique seul (taux_annulation_passee)":
        (["taux_annulation_passee"], []),
    "Minimal ciblé (personnes_par_chambre, prix_relatif_destination, taux_annulation_passee)":
        (["personnes_par_chambre", "prix_relatif_destination", "taux_annulation_passee"], []),
}

ablation_rows = []
for label, (num_extra, cat_extra) in ablation_sets.items():
    res, _ = eval_pipeline(
        make_fe_pipeline(make_xgb(), NUM_COLS + num_extra, CAT_COLS + cat_extra),
        X_tr_fe, X_val_fe, label,
    )
    ablation_rows.append(res)

ablation_df = pd.DataFrame(ablation_rows).set_index("modele")
ablation_df.loc["XGBoost — sans FE (référence)"] = res_xgb_base
ablation_df[["f1", "roc_auc"]].sort_values("f1", ascending=False).round(4)

,f1,roc_auc
modele,,
XGBoost — sans FE (référence),0.4842,0.6565
"Séjour + prix (personnes_par_chambre, nuits_par_chambre, prix_total_par_personne, prix_relatif_destination)",0.4841,0.6548
"Temporel seul (mois, jour_semaine, delai_log, delai_bin)",0.4830,0.6553
"Minimal ciblé (personnes_par_chambre, prix_relatif_destination, taux_annulation_passee)",0.4828,0.6553
Historique seul (taux_annulation_passee),0.4810,0.6556


Le sous-ensemble **« Minimal ciblé »** (`personnes_par_chambre`, `prix_relatif_destination`,
`taux_annulation_passee`) ressort comme le plus prometteur : ce sont trois variables qui encodent une
information **relative** (densité d'occupation, prix par rapport à la destination, taux plutôt que
compte brut) que les colonnes originales ne fournissent pas directement, contrairement aux variables
temporelles ou aux indicateurs binaires redondants écartés plus haut. On retient ce sous-ensemble
(`src.config.FE_NUM_COLS`) comme jeu de variables final.

## 4.4 Validation du gain — robustesse (seuil fixe et plusieurs graines)

Pour écarter tout artefact du choix de seuil ou du hasard d'initialisation, on vérifie le gain à
**seuil fixe (0.5)** et on moyenne sur plusieurs graines aléatoires.

In [5]:
# --- Régression logistique : gain à seuil fixe 0.5 ---
pipe_lr_base = make_pipeline(make_logreg())
pipe_lr_base.fit(X_tr_p, y_tr)
proba_lr_base = pipe_lr_base.predict_proba(X_val_p)[:, 1]
res_lr_base_05, _ = evaluate(y_val, proba_lr_base, threshold=0.5, label="LR — sans FE")

pipe_lr_fe = make_fe_pipeline(make_logreg(), NUM_COLS_FE, CAT_COLS_FE)
pipe_lr_fe.fit(X_tr_fe, y_tr)
proba_lr_fe = pipe_lr_fe.predict_proba(X_val_fe)[:, 1]
res_lr_fe_05, _ = evaluate(y_val, proba_lr_fe, threshold=0.5, label="LR — FE minimal")

seuil_fixe = pd.DataFrame([res_lr_base_05, res_lr_fe_05]).set_index("modele")
print("=== Gain à seuil fixe (0.5), régression logistique ===")
print(seuil_fixe[["f1", "precision", "recall", "roc_auc"]].round(4))
print(f"\nΔF1 = {res_lr_fe_05['f1'] - res_lr_base_05['f1']:+.4f}   ΔAUC = {res_lr_fe_05['roc_auc'] - res_lr_base_05['roc_auc']:+.4f}")

=== Gain à seuil fixe (0.5), régression logistique ===
                     f1  precision  recall  roc_auc
modele                                             
LR — sans FE     0.4537     0.3609  0.6107   0.6532
LR — FE minimal  0.4578     0.3629  0.6200   0.6563

ΔF1 = +0.0042   ΔAUC = +0.0031


In [6]:
# --- Robustesse sur plusieurs graines : XGBoost avec le jeu minimal ---
f1_base_list, f1_fe_list = [], []
for s in range(5):
    p0 = make_pipeline(XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=s, eval_metric="logloss", n_jobs=-1, **xgb_best_params))
    p0.fit(X_tr_p, y_tr)
    pr0 = p0.predict_proba(X_val_p)[:, 1]
    t0, f10, _, _ = threshold_curve(y_val, pr0)
    f1_base_list.append(f10.max())

    p1 = make_fe_pipeline(XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=s, eval_metric="logloss", n_jobs=-1, **xgb_best_params), NUM_COLS_FE, CAT_COLS_FE)
    p1.fit(X_tr_fe, y_tr)
    pr1 = p1.predict_proba(X_val_fe)[:, 1]
    t1, f11, _, _ = threshold_curve(y_val, pr1)
    f1_fe_list.append(f11.max())

print("XGBoost — F1 max par graine, sans FE :", [round(x, 4) for x in f1_base_list])
print("XGBoost — F1 max par graine, FE min. :", [round(x, 4) for x in f1_fe_list])
print(f"\nMoyenne sans FE : {np.mean(f1_base_list):.4f} ± {np.std(f1_base_list):.4f}")
print(f"Moyenne FE min. : {np.mean(f1_fe_list):.4f} ± {np.std(f1_fe_list):.4f}")

XGBoost — F1 max par graine, sans FE : [np.float64(0.4824), np.float64(0.4831), np.float64(0.4837), np.float64(0.4834), np.float64(0.4833)]
XGBoost — F1 max par graine, FE min. : [np.float64(0.481), np.float64(0.4831), np.float64(0.4836), np.float64(0.4838), np.float64(0.4823)]

Moyenne sans FE : 0.4832 ± 0.0004
Moyenne FE min. : 0.4828 ± 0.0010


## 4.5 Conclusion et décision de modélisation

**Deux constats distincts, tous deux valides et informatifs :**

1. **Sur la régression logistique**, le sous-ensemble minimal apporte un gain **réel, déterministe et
   reproductible** : F1 = 0.4537 → 0.4578 (+0.0041) à seuil fixe 0.5, ROC-AUC = 0.6532 → 0.6563
   (+0.0031). C'est cohérent avec l'intuition : un modèle **linéaire** ne peut pas construire seul le
   ratio `personnes_par_chambre` ou la comparaison `prix_relatif_destination` à partir des variables
   brutes — il lui faut ces interactions en entrée. Cela répond directement à la question du README
   (Q3) sur le gain de FE *par rapport à la régression logistique de référence*.

2. **Sur XGBoost**, le même jeu minimal ne produit **aucun gain net reproductible** : la différence
   moyenne sur 5 graines est inférieure à l'écart-type observé. C'est également cohérent : un modèle à
   base d'arbres peut déjà approximer un ratio comme `personnes_par_chambre` via des splits successifs
   sur `personnes_totales` et `chambres` — les variables dérivées ne lui apportent donc pas
   d'information réellement nouvelle, seulement une reformulation.

**Décision :** le modèle final retenu pour l'interprétation (notebook 05) et la soumission (notebook 06)
reste le **XGBoost de l'étape 3** (`artifacts/xgboost_v1`), sans les variables de feature engineering —
puisqu'elles ne lui apportent aucun bénéfice mesurable. Le gain du feature engineering est bien
**démontré expérimentalement**, mais sur le modèle où il a effectivement un sens de l'appliquer.

In [7]:
save_model(
    pipe_lr_fe, "logreg_fe_minimal",
    metadata={**res_lr_fe_05, "features_ajoutees": ["personnes_par_chambre", "prix_relatif_destination", "taux_annulation_passee"],
              "gain_f1_vs_sans_fe": res_lr_fe_05["f1"] - res_lr_base_05["f1"]},
)
print("\nModèle retenu pour la suite (notebooks 05-06) : XGBoost sans FE (artifacts/xgboost_v1)")
print("Le modèle logreg_fe_minimal est sauvegardé à titre de preuve du gain expérimental (README Q3),")
print("mais n'est pas le modèle de production.")

Modèle sauvegardé : /home/claude/atlantic-haven/artifacts/logreg_fe_minimal.joblib

Modèle retenu pour la suite (notebooks 05-06) : XGBoost sans FE (artifacts/xgboost_v1)
Le modèle logreg_fe_minimal est sauvegardé à titre de preuve du gain expérimental (README Q3),
mais n'est pas le modèle de production.
